# Прогноз времени на ДЗ — эталон (исправленная версия)

Исправлено:
1. Опечатка `+1 np...` → `+ np.random.normal(...)`.
2. Отступы в функции `predict`.
3. Убран нерабочий `if name == "main"` → просто `demo.launch()`.
4. Модель обёрнута в `Pipeline` (стандарт курса).

⚠️ Данные синтетические — это учебная демонстрация. Для финального проекта лучше подставить реальные данные.

In [ ]:
import gradio as gr
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# --- синтетические данные (учебная демонстрация) ---
np.random.seed(42)
n_samples = 100

subjects = np.random.randint(1, 7, size=n_samples)
difficulty = np.random.randint(1, 6, size=n_samples)

# ФИКС 1: правильный "+ шум"
time_spent = subjects * 25 + difficulty * 15 + np.random.normal(0, 10, n_samples)
time_spent = np.clip(time_spent, 15, 300)

X = pd.DataFrame({"subjects": subjects, "difficulty": difficulty})
y = time_spent

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- baseline: всегда средняя (отметка на стене) ---
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
mae_baseline = mean_absolute_error(y_test, baseline.predict(X_test))

# ФИКС 4: модель в Pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])
model.fit(X_train, y_train)
mae_model = mean_absolute_error(y_test, model.predict(X_test))

print(f"Baseline MAE: {mae_baseline:.2f} min")
print(f"Model MAE:    {mae_model:.2f} min")


# ФИКС 2: правильные отступы в функции
def predict(subjects_count, difficulty_level):
    input_data = pd.DataFrame(
        [[subjects_count, difficulty_level]],
        columns=["subjects", "difficulty"],
    )
    prediction = model.predict(input_data)[0]
    return f"{max(10, int(round(prediction)))} минут"


demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Slider(minimum=1, maximum=6, step=1, value=3, label="Количество предметов"),
        gr.Slider(minimum=1, maximum=5, step=1, value=3, label="Уровень сложности (1-5)"),
    ],
    outputs=gr.Textbox(label="Прогнозируемое время на ДЗ"),
    title="Прогноз времени на домашнее задание",
    description="Модель машинного обучения на базе Linear Regression",
)

# ФИКС 3: без нерабочего if __name__ — просто запуск
demo.launch(share=True)